In [30]:
import confnotebook

In [31]:
from pathlib import Path

source = Path("../examples/test/bag_date/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 127113
[1] 127116
[2] Сверка БайкАкв от 03.06.26
[3] Ситилинк


In [32]:
IDX_FILE = 2

In [33]:
import base64
import time

import requests

BASE_URL = "http://127.0.0.1:8001"
pdf_path = files[IDX_FILE]

document_b64 = base64.b64encode(pdf_path.read_bytes()).decode()
resp = requests.post(f"{BASE_URL}/send_reconciliation_act", json={"document": document_b64})

if not resp.ok:
    print(resp.text)


resp.raise_for_status()

process_id = resp.json()["process_id"]
print(f"process_id: {process_id}")

while True:
    resp = requests.post(f"{BASE_URL}/process_status", json={"process_id": process_id})
    if resp.status_code == 200:
        data = resp.json()
        print(f"seller: {data['seller']}")
        print(f"buyer:  {data['buyer']}")
        print(f"debit:  {len(data['debit'])} entries")
        print(f"credit: {len(data['credit'])} entries")
        break
    elif resp.status_code == 201:
        print("processing...")
        time.sleep(3)
    else:
        raise RuntimeError(resp.json())

process_id: 966369c3-4799-410b-8788-6c196356d9cd
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
seller: ВАЛЬКОВ АЛЕКСЕЙ АЛЕКСАНДРОВИЧ, ИП
buyer:  ПК  БАЙКАЛ АКВА, ООО
debit:  33 entries
credit: 33 entries


In [34]:
comments = """
            По данным АО "РУСАЛ Новокузнецк" на 30.09.2023
            задолженность в пользу АО "РУСАЛ Новокузнецк"
            составляет 13 755 023,24 руб.
            С разногласиями, протокол разногласий прилагается.
            Акт сверки проверен ОУФО ОЦО, ООО "РЦУ".
            Исполнитель: Воробьева Оксана Евгеньевна
            Дата:29.01.2025
            """

fill_resp = requests.post(f"{BASE_URL}/fill_reconciliation_act", json={
    "process_id": process_id,
    "comments": comments,
    "debit": data["debit"],
    "credit": data["credit"],
})
if not fill_resp.ok:
    print(fill_resp.json())
else:
    fill_resp.raise_for_status()

    out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
    Path(out_path).write_bytes(base64.b64decode(fill_resp.json()["document"]))
    print(out_path)

../examples/output/Сверка БайкАкв от 03.06.26_filled.pdf
